In [1]:
# import statements
from oqd_core.interface.analog import *
from oqd_core.interface.analog.register import QuantumRef, ClassicalRef, QuantumRegister, ClassicalRegister
from oqd_core.compiler.analog.passes.resolve import resolve_analog_declarations
from oqd_core.interface.math import MathRef, MathNum, MathStr
from oqd_compiler_infrastructure import Post, PrettyPrint

printer = Post(PrettyPrint())

In [2]:
# Declare quantum and classical registers via `declarations`, and an operator variable
# via `OperatorDeclaration`

X, Z = PauliX(), PauliZ()

circuit = AnalogCircuit(
    declarations=[
        QuantumDeclaration(name="q", size=3),
        ClassicalDeclaration(name="c", size=3),
        OperatorDeclaration(name="H", operator=X * 0.5),
    ],
    sequence=[
        Evolve(gate=AnalogGate(hamiltonian=X * 0.5), duration=2.0),
        Evolve(gate=AnalogGate(hamiltonian=Z @ Z * 0.3), duration=1.0),
    ]
)

print(printer(circuit))

AnalogCircuit
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(3)
    - 1: ClassicalDeclaration
      - name: str(c)
      - size: int(3)
    - 2: OperatorDeclaration
      - name: str(H)
      - operator: OperatorScalarMul
        - op: PauliX
        - expr: MathNum
          - value: float(0.5)
  - sequence: list
    - 0: Evolve
      - key: str(evolve)
      - duration: float(2.0)
      - gate: AnalogGate
        - hamiltonian: OperatorScalarMul
          - op: PauliX
          - expr: MathNum
            - value: float(0.5)
    - 1: Evolve
      - key: str(evolve)
      - duration: float(1.0)
      - gate: AnalogGate
        - hamiltonian: OperatorScalarMul
          - op: OperatorKron
            - op1: PauliZ
            - op2: PauliZ
          - expr: MathNum
            - value: float(0.3)
  - n_qreg: NoneType(None)
  - n_qmode: NoneType(None)


In [3]:
# Use `IfElse` with a `ClassicalRef` condition. The condition references a declared
# classical register by name. After resolution, the symbolic reference is replaced
# with a concrete `ClassicalRegister`.

circuit = AnalogCircuit(
    declarations=[
        QuantumDeclaration(name="q", size=3),
        ClassicalDeclaration(name="c", size=3),
    ],
    sequence=[
        Evolve(gate=AnalogGate(hamiltonian=X * 0.5), duration=2.0),
        IfElse(
            condition=RegisterNonZero(creg=ClassicalRef(name="c")),
            then_branch=[
                Evolve(gate=AnalogGate(hamiltonian=Z * 0.1), duration=1.0),
            ],
            else_branch=[
                Evolve(gate=AnalogGate(hamiltonian=X * 0.2), duration=1.5),
            ],
        ),
    ]
)

print(printer(circuit))

AnalogCircuit
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(3)
    - 1: ClassicalDeclaration
      - name: str(c)
      - size: int(3)
  - sequence: list
    - 0: Evolve
      - key: str(evolve)
      - duration: float(2.0)
      - gate: AnalogGate
        - hamiltonian: OperatorScalarMul
          - op: PauliX
          - expr: MathNum
            - value: float(0.5)
    - 1: IfElse
      - key: str(if_else)
      - condition: RegisterNonZero
        - creg: ClassicalRef
          - name: str(c)
          - index: NoneType(None)
      - then_branch: list
        - 0: Evolve
          - key: str(evolve)
          - duration: float(1.0)
          - gate: AnalogGate
            - hamiltonian: OperatorScalarMul
              - op: PauliZ
              - expr: MathNum
                - value: float(0.1)
      - else_branch: list
        - 0: Evolve
          - key: str(evolve)
          - duration: float(1.5)
       

In [4]:
resolved = resolve_analog_declarations(circuit)

print(printer(resolved))

AnalogCircuit
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(3)
    - 1: ClassicalDeclaration
      - name: str(c)
      - size: int(3)
  - sequence: list
    - 0: Evolve
      - key: str(evolve)
      - duration: float(2.0)
      - gate: AnalogGate
        - hamiltonian: OperatorScalarMul
          - op: PauliX
          - expr: MathNum
            - value: float(0.5)
    - 1: IfElse
      - key: str(if_else)
      - condition: RegisterNonZero
        - creg: ClassicalRegister
          - id: str(c)
          - reg: list
            - 0: ClassicalBit
              - id: str(c)
              - index: int(0)
            - 1: ClassicalBit
              - id: str(c)
              - index: int(1)
            - 2: ClassicalBit
              - id: str(c)
              - index: int(2)
      - then_branch: list
        - 0: Evolve
          - key: str(evolve)
          - duration: float(1.0)
          - gate: AnalogGate
 

In [5]:
# Use `While` with a `ClassicalRef` condition to represent a classically-controlled loop.
circuit = AnalogCircuit(
    declarations=[
        QuantumDeclaration(name="q", size=2),
        ClassicalDeclaration(name="flag", size=1),
    ],
    sequence=[
        Evolve(gate=AnalogGate(hamiltonian=X * 0.5), duration=1.0),
        While(
            condition=RegisterNonZero(creg=ClassicalRef(name="flag")),
            body=[
                Evolve(gate=AnalogGate(hamiltonian=Z * 0.3), duration=0.5),
            ],
        ),
    ]
)

resolved = resolve_analog_declarations(circuit)

print(printer(resolved))

AnalogCircuit
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(2)
    - 1: ClassicalDeclaration
      - name: str(flag)
      - size: int(1)
  - sequence: list
    - 0: Evolve
      - key: str(evolve)
      - duration: float(1.0)
      - gate: AnalogGate
        - hamiltonian: OperatorScalarMul
          - op: PauliX
          - expr: MathNum
            - value: float(0.5)
    - 1: While
      - key: str(while)
      - condition: RegisterNonZero
        - creg: ClassicalRegister
          - id: str(flag)
          - reg: list
            - 0: ClassicalBit
              - id: str(flag)
              - index: int(0)
      - body: list
        - 0: Evolve
          - key: str(evolve)
          - duration: float(0.5)
          - gate: AnalogGate
            - hamiltonian: OperatorScalarMul
              - op: PauliZ
              - expr: MathNum
                - value: float(0.3)
  - n_qreg: NoneType(None)
  - n_qmode

In [6]:
# The `qreg`/`creg` and `declarations` entry points are independent.
q = QuantumRegister(id="r", reg=2)

circuit = AnalogCircuit(
    qreg=[q],
    declarations=[
        ClassicalDeclaration(name="c", size=2),
    ],
    sequence=[
        Evolve(gate=AnalogGate(hamiltonian=X * 0.5), duration=2.0),
        IfElse(
            condition=RegisterNonZero(creg=ClassicalRef(name="c")),
            then_branch=[
                Evolve(gate=AnalogGate(hamiltonian=Z * 0.1), duration=1.0),
            ],
        ),
    ]
)

resolved = resolve_analog_declarations(circuit)

print(printer(resolved))

AnalogCircuit
  - qreg: list
    - 0: QuantumRegister
      - id: str(r)
      - reg: list
        - 0: QuantumBit
          - id: str(r)
          - index: int(0)
        - 1: QuantumBit
          - id: str(r)
          - index: int(1)
  - creg: list
  - declarations: list
    - 0: ClassicalDeclaration
      - name: str(c)
      - size: int(2)
  - sequence: list
    - 0: Evolve
      - key: str(evolve)
      - duration: float(2.0)
      - gate: AnalogGate
        - hamiltonian: OperatorScalarMul
          - op: PauliX
          - expr: MathNum
            - value: float(0.5)
    - 1: IfElse
      - key: str(if_else)
      - condition: RegisterNonZero
        - creg: ClassicalRegister
          - id: str(c)
          - reg: list
            - 0: ClassicalBit
              - id: str(c)
              - index: int(0)
            - 1: ClassicalBit
              - id: str(c)
              - index: int(1)
      - then_branch: list
        - 0: Evolve
          - key: str(evolve)
          - 

In [7]:
# Boolean assignment: declare a condition, reference it in IfElse

circuit = AnalogCircuit(
    declarations=[
        QuantumDeclaration(name="q", size=2),
        ClassicalDeclaration(name="c", size=2),
        BoolDeclaration(
            name="flag",
            expr=RegisterNonZero(creg=ClassicalRef(name="c")),
        ),
    ],
    sequence=[
        Evolve(gate=AnalogGate(hamiltonian=X * 0.5), duration=2.0),
        IfElse(
            condition=BoolRef(name="flag"),
            then_branch=[Evolve(gate=AnalogGate(hamiltonian=Z * 0.1), duration=1.0)],
            else_branch=[Evolve(gate=AnalogGate(hamiltonian=X * 0.2), duration=1.5)],
        ),
    ]
)

print(printer(circuit))

AnalogCircuit
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(2)
    - 1: ClassicalDeclaration
      - name: str(c)
      - size: int(2)
    - 2: BoolDeclaration
      - name: str(flag)
      - expr: RegisterNonZero
        - creg: ClassicalRef
          - name: str(c)
          - index: NoneType(None)
  - sequence: list
    - 0: Evolve
      - key: str(evolve)
      - duration: float(2.0)
      - gate: AnalogGate
        - hamiltonian: OperatorScalarMul
          - op: PauliX
          - expr: MathNum
            - value: float(0.5)
    - 1: IfElse
      - key: str(if_else)
      - condition: BoolRef
        - name: str(flag)
      - then_branch: list
        - 0: Evolve
          - key: str(evolve)
          - duration: float(1.0)
          - gate: AnalogGate
            - hamiltonian: OperatorScalarMul
              - op: PauliZ
              - expr: MathNum
                - value: float(0.1)
      - else_branc

In [8]:
resolved = resolve_analog_declarations(circuit)
print(printer(resolved))

AnalogCircuit
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(2)
    - 1: ClassicalDeclaration
      - name: str(c)
      - size: int(2)
    - 2: BoolDeclaration
      - name: str(flag)
      - expr: RegisterNonZero
        - creg: ClassicalRef
          - name: str(c)
          - index: NoneType(None)
  - sequence: list
    - 0: Evolve
      - key: str(evolve)
      - duration: float(2.0)
      - gate: AnalogGate
        - hamiltonian: OperatorScalarMul
          - op: PauliX
          - expr: MathNum
            - value: float(0.5)
    - 1: IfElse
      - key: str(if_else)
      - condition: RegisterNonZero
        - creg: ClassicalRegister
          - id: str(c)
          - reg: list
            - 0: ClassicalBit
              - id: str(c)
              - index: int(0)
            - 1: ClassicalBit
              - id: str(c)
              - index: int(1)
      - then_branch: list
        - 0: Evolve
          - 

In [9]:
# Compound condition: BoolAnd of two BitEquals
circuit = AnalogCircuit(
    declarations=[
        QuantumDeclaration(name="q", size=2),
        ClassicalDeclaration(name="c", size=2),
    ],
    sequence=[
        IfElse(
            condition=BoolAnd(
                left=BitEquals(creg=ClassicalRef(name="c"), index=0, value=1),
                right=BoolNot(expr=BitEquals(creg=ClassicalRef(name="c"), index=1, value=1)),
            ),
            then_branch=[Evolve(gate=AnalogGate(hamiltonian=Z * 0.1), duration=1.0)],
            else_branch=[],
        ),
    ]
)
resolved = resolve_analog_declarations(circuit)
print(printer(resolved))

AnalogCircuit
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(2)
    - 1: ClassicalDeclaration
      - name: str(c)
      - size: int(2)
  - sequence: list
    - 0: IfElse
      - key: str(if_else)
      - condition: BoolAnd
        - left: BitEquals
          - creg: ClassicalRegister
            - id: str(c)
            - reg: list
              - 0: ClassicalBit
                - id: str(c)
                - index: int(0)
              - 1: ClassicalBit
                - id: str(c)
                - index: int(1)
          - index: int(0)
          - value: int(1)
        - right: BoolNot
          - expr: BitEquals
            - creg: ClassicalRegister
              - id: str(c)
              - reg: list
                - 0: ClassicalBit
                  - id: str(c)
                  - index: int(0)
                - 1: ClassicalBit
                  - id: str(c)
                  - index: int(1)
            

In [10]:
# MathExpr assignment: declare omega, use MathRef in operator
circuit = AnalogCircuit(
    declarations=[
        QuantumDeclaration(name="q", size=2),
        MathExprDeclaration(name="omega", expr=MathNum(value=0.5)),
    ],
    sequence=[
        Evolve(
            gate=AnalogGate(
                hamiltonian=X * MathRef(name="omega"),
            ),
            duration=2.0,
        ),
    ]
)
resolved = resolve_analog_declarations(circuit)
print(printer(resolved))

AnalogCircuit
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(2)
    - 1: MathExprDeclaration
      - name: str(omega)
      - expr: MathNum
        - value: float(0.5)
  - sequence: list
    - 0: Evolve
      - key: str(evolve)
      - duration: float(2.0)
      - gate: AnalogGate
        - hamiltonian: OperatorScalarMul
          - op: PauliX
          - expr: MathNum
            - value: float(0.5)
  - n_qreg: NoneType(None)
  - n_qmode: NoneType(None)
